# Curtailment model — batch scoring (spec 11)

Scores every wind/solar plant-hour in `ampera_gold.curtailment_features_hourly` with the UC model
`curtailment_expected_cf@champion`, flags curtailment (`expected CF >= 10%` and `actual < Y x expected`,
`Y` from the model version tags), values the lost energy at the hour's PLD and writes:
- `ampera_gold.curtailment_hourly` — lakehouse, for Genie and analysis
- `ampera_gold.curtailment_daily` — row-filtered per client (spec 17), plus its unfiltered copy
  `ampera_serving.curtailment_daily`, synced to Lakebase for the app

Runs in the refresh job after the medallion and before the Lakebase syncs.

In [ ]:
%pip install -q "mlflow>=3.1" scikit-learn

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
dbutils.widgets.text("catalog", "ampera_catalog")  # noqa: F821
CAT = dbutils.widgets.get("catalog")  # noqa: F821
FEAT = f"{CAT}.ampera_gold.curtailment_features_hourly"
MODEL = f"{CAT}.ampera_gold.curtailment_expected_cf"

In [ ]:
import os
import sys

import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F

sys.path.append(os.path.abspath("../src"))
import curtailment_lib as C  # noqa: E402  flag + valuation rules (tests/test_curtailment.py)

mlflow.set_registry_uri("databricks-uc")
mv = MlflowClient().get_model_version_by_alias(MODEL, "champion")
Y = {"EOL": float(mv.tags["y_eol"]), "SOL": float(mv.tags["y_sol"])}
MIN_CF = float(mv.tags["min_expected_cf"])
print(f"{MODEL} v{mv.version}  Y={Y}  min_expected_cf={MIN_CF}")
expected_cf = mlflow.pyfunc.spark_udf(spark, f"models:/{MODEL}@champion", result_type="double",  # noqa: F821
                                      env_manager="local")

ampera_catalog.ampera_gold.curtailment_expected_cf v3  Y={'EOL': 0.75, 'SOL': 1.0}  min_expected_cf=0.1
2026/10/27 23:13:17 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.
2026/10/27 23:13:17 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [ ]:
cols = ["fonte", "plant_id", "uf", "hora", "mes", "vento_100m_ms", "irradiancia_wm2", "temp_c", "lat", "lon",
        "vento_ms_medido_ons", "irradiancia_wm2_medida_ons"]
f = spark.table(FEAT)  # noqa: F821
y = F.when(F.col("fonte") == "EOL", F.lit(Y["EOL"])).otherwise(F.lit(Y["SOL"]))
s = (f.withColumn("cf_esperado", expected_cf(*cols))
      .withColumn("mwh_esperado", F.col("cf_esperado") * F.col("capacity_mw"))
      .withColumn("restrito_modelo", C.flagged(F.col("cf_esperado"), F.col("fator_capacidade"), y, MIN_CF))
      .withColumn("mwh_restrito", F.when(F.col("restrito_modelo"), F.greatest(F.col("mwh_esperado") - F.col("mwh"), F.lit(0.0)))
                                   .otherwise(F.lit(0.0)))  # = C.lost_mwh
      .withColumn("valor_rs_perdido", C.value_rs(F.col("mwh_restrito"), F.col("pld_rs_mwh")))
      .select("ts", "gen_date", "plant_id", "nome", "uf", "submercado", "fonte", "conjunto_key", "capacity_mw",
              "mwh", "mwh_esperado", "restrito_modelo", "restrito_ons", "mwh_estimado_ons", "mwh_restrito",
              "pld_rs_mwh", "valor_rs_perdido", F.lit(int(mv.version)).alias("model_version")))
(s.write.mode("overwrite").option("overwriteSchema", "true")
  .clusterBy("gen_date", "plant_id").saveAsTable(f"{CAT}.ampera_gold.curtailment_hourly"))
spark.sql(f"""COMMENT ON TABLE {CAT}.ampera_gold.curtailment_hourly IS
  'Gold (ML): hourly expected vs actual MWh per wind/solar plant; restrito_modelo = model curtailment flag, restrito_ons = ONS flag; mwh_restrito valued at PLD (valor_rs_perdido). Spec 11.'""")

In [ ]:
h = spark.table(f"{CAT}.ampera_gold.curtailment_hourly")  # noqa: F821
d = (h.groupBy(F.col("gen_date").alias("dia"), "plant_id")
      .agg(F.max("nome").alias("nome"), F.max("uf").alias("uf"), F.max("submercado").alias("submercado"),
           F.max("fonte").alias("fonte"), F.max("conjunto_key").alias("conjunto_key"),
           F.max("capacity_mw").alias("capacidade_mw"),
           F.sum("mwh").alias("mwh"), F.sum("mwh_esperado").alias("mwh_esperado"),
           F.sum(F.col("restrito_modelo").cast("int")).alias("horas_restritas_modelo"),
           F.sum(F.col("restrito_ons").cast("int")).alias("horas_restritas_ons"),
           F.sum("mwh_restrito").alias("mwh_restrito"), F.sum("valor_rs_perdido").alias("valor_rs_perdido")))
# Two copies (spec 17): the unfiltered serving copy is the Lakebase sync source (synced tables refuse a
# row-filtered source); the gold copy carries the client row filter for Genie and SQL readers.
SERV = f"{CAT}.ampera_serving.curtailment_daily"
(d.write.mode("overwrite").option("overwriteSchema", "true")
  .option("delta.enableChangeDataFeed", "true")
  .clusterBy("dia", "plant_id").saveAsTable(SERV))
spark.sql(f"COMMENT ON TABLE {SERV} IS 'Serving: unfiltered copy of ampera_gold.curtailment_daily, synced to Lakebase. Spec 17.'")  # noqa: F821
spark.sql(f"ALTER TABLE {SERV} ALTER COLUMN dia SET NOT NULL")  # noqa: F821
spark.sql(f"ALTER TABLE {SERV} ALTER COLUMN plant_id SET NOT NULL")  # noqa: F821
spark.sql(f"""CREATE OR REPLACE TABLE {CAT}.ampera_gold.curtailment_daily
  CLUSTER BY (dia, plant_id)
  COMMENT 'Gold (ML): daily curtailment per wind/solar plant — expected vs actual MWh, curtailed hours (model and ONS), MWh and R$ lost at PLD. Clients see only their plants (row filter). Spec 11, 17.'
  WITH ROW FILTER {CAT}.ampera_gold.client_plant_filter ON (plant_id)
  AS SELECT * FROM {SERV}""")  # noqa: F821

 ## Results

In [ ]:
def show(sql):
    print(spark.sql(sql).toPandas().to_string(index=False))  # noqa: F821


show(f"""SELECT fonte, count(DISTINCT plant_id) AS usinas, round(sum(mwh)/1e6, 2) AS geracao_twh,
           round(sum(mwh_restrito)/1e3, 1) AS restrito_gwh, round(sum(valor_rs_perdido)/1e6, 1) AS perdido_rs_milhoes,
           round(100*sum(mwh_restrito)/sum(mwh_esperado), 2) AS pct_do_esperado
    FROM {CAT}.ampera_gold.curtailment_daily GROUP BY fonte ORDER BY fonte""")
print("\nModel flag vs ONS flag, all labeled plant-hours (both periods):")
show(f"""SELECT fonte, restrito_ons, restrito_modelo, count(*) AS plant_hours
    FROM {CAT}.ampera_gold.curtailment_hourly WHERE restrito_ons IS NOT NULL
    GROUP BY ALL ORDER BY ALL""")
print("\nCoverage: plant-hours with no ONS curtailment flag (ONS's Oct-Dec 2024 solar files carry none) — the model still scores them:")
show(f"""SELECT fonte, count_if(restrito_ons IS NULL) AS sem_rotulo_ons, count(*) AS total,
           round(sum(CASE WHEN restrito_ons IS NULL THEN mwh_restrito END)/1e3, 1) AS restrito_gwh_sem_rotulo
    FROM {CAT}.ampera_gold.curtailment_hourly GROUP BY fonte ORDER BY fonte""")
print("\nTop 10 plants by R$ lost:")
show(f"""SELECT plant_id, max(nome) AS nome, max(uf) AS uf, max(fonte) AS fonte,
           round(sum(mwh_restrito)/1e3, 1) AS restrito_gwh, round(sum(valor_rs_perdido)/1e6, 2) AS perdido_rs_milhoes
    FROM {CAT}.ampera_gold.curtailment_daily GROUP BY plant_id ORDER BY perdido_rs_milhoes DESC LIMIT 10""")

fonte  usinas  geracao_twh  restrito_gwh  perdido_rs_milhoes  pct_do_esperado
  EOL    1052       213.55       46071.1              6592.1            18.26
  SOL     546        64.51       23861.4              2596.0            27.97

Model flag vs ONS flag, all labeled plant-hours (both periods):
fonte  restrito_ons  restrito_modelo  plant_hours
  EOL         False            False      9331567
  EOL         False             True      1416632
  EOL          True            False      3852785
  EOL          True             True      3494877
  SOL         False            False      4595823
  SOL         False             True       528143
  SOL          True            False       430645
  SOL          True             True      1755717

Coverage: plant-hours with no ONS curtailment flag (ONS's Oct-Dec 2024 solar files carry none) — the model still scores them:
fonte  sem_rotulo_ons    total  restrito_gwh_sem_rotulo
  EOL            1656 18097517                     55.4
  SOL       